# Evaluation run

Serves the model with vLLM on the GPU and runs the warehouse, dbt, the MCP server and the agents alongside it.
Needs an A100 (80 GB) with high RAM. Runs are traced in Langfuse when the `LANGFUSE_*` secrets are set.

Results are written to Drive after each incident; incidents that already have a result are skipped.

In [1]:
import os, subprocess

repo = "/content/responder"
if os.path.exists(repo):
    subprocess.run(["git", "-C", repo, "pull", "-q"], check=True)
else:
    subprocess.run(["git", "clone", "-q", "https://github.com/arihaaant/data-incident-responder.git", repo], check=True)
print("code at", subprocess.run(["git", "-C", repo, "log", "--oneline", "-1"], capture_output=True, text=True).stdout)
%cd /content/responder
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q vllm
!pip uninstall -y -q torchaudio
!pip install -q -e .

code at c60e721 evaluation notebook: every incident, three runs per system

/content/responder
name, memory.total [MiB]
NVIDIA A100-SXM4-80GB, 81920 MiB
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.6/320.6 MB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 105.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.7/211.7 kB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.7/33.7 MB 81.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 426.0/426.0 kB 38.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 106.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 782.6/782.6 kB 54.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.

In [2]:
import os
from google.colab import userdata

for key in ["LANGFUSE_PUBLIC_KEY", "LANGFUSE_SECRET_KEY", "LANGFUSE_BASE_URL"]:
    try:
        os.environ[key] = userdata.get(key)
    except Exception:
        pass
print("tracing on" if os.getenv("LANGFUSE_PUBLIC_KEY") else "tracing off")

tracing on


## Model server

In [3]:
import subprocess, time, urllib.request

model = "openai/gpt-oss-120b"
log = open("/content/vllm.log", "w")
server = subprocess.Popen(
    ["vllm", "serve", model, "--port", "8000", "--max-model-len", "32768", "--gpu-memory-utilization", "0.9",
     "--enable-auto-tool-choice", "--tool-call-parser", "openai"],
    stdout=log, stderr=subprocess.STDOUT)

t0 = time.time()
while True:
    try:
        urllib.request.urlopen("http://localhost:8000/v1/models", timeout=5)
        break
    except Exception:
        if server.poll() is not None or time.time() - t0 > 1800:
            print(open("/content/vllm.log").read()[-3000:])
            raise RuntimeError("vllm did not start")
        time.sleep(15)
print(f"model up after {(time.time() - t0) / 60:.1f} min")
os.environ["LLM_PROVIDER"], os.environ["LLM_MODEL"] = "vllm", model

model up after 7.8 min


## Warehouse and tests

In [4]:
import subprocess, sys
!python -m responder.warehouse --path kit/source.duckdb
tests = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests"], capture_output=True, text=True)
print(tests.stdout[-1500:])
assert tests.returncode == 0, "test suite failed"

built kit/source.duckdb
...........................................................              [100%]
59 passed in 35.13s



## Responder and baselines on every incident, three runs each

In [6]:
from google.colab import drive

drive.mount("/content/drive")
run_name = "final"
out = f"/content/drive/MyDrive/data-incident-responder/{run_name}"
os.makedirs(out, exist_ok=True)

Mounted at /content/drive


In [7]:
print("=== rules ===")
!python -m responder.run --system rules --workers 6 --out {out}/rules
for r in range(1, 4):
    for system in ["multi", "single"]:
        print(f"\n=== {system}, run {r} ===")
        !python -m responder.run --system {system} --workers 8 --out {out}/{system}-{r}

Streaming output truncated to the last 5000 lines.
                             CallToolRequest                                    
                    INFO     Processing request of type            ]8;id=635909;file:///usr/local/lib/python3.13/dist-packages/mcp/server/lowlevel/server.py\server.py]8;;\:]8;id=506560;file:///usr/local/lib/python3.13/dist-packages/mcp/server/lowlevel/server.py#733\733]8;;\
                             ListToolsRequest                                   
[10/12/26 00:28:42] INFO     Processing request of type            ]8;id=908401;file:///usr/local/lib/python3.13/dist-packages/mcp/server/lowlevel/server.py\server.py]8;;\:]8;id=772268;file:///usr/local/lib/python3.13/dist-packages/mcp/server/lowlevel/server.py#733\733]8;;\
                             CallToolRequest                                    
                    INFO     Processing request of type            ]8;id=630046;file:///usr/local/lib/python3.13/dist-packages/mcp/server/l

## Results

In [8]:
for name in ["rules"] + [f"{s}-{r}" for r in range(1, 4) for s in ["multi", "single"]]:
    print(f"\n=== {name} ===")
    !python -m responder.report {out}/{name}


=== rules ===
                   n    root   cause  action  resolved  tokens
overall           63   27.0%   71.4%   33.3%     33.3%       0
duplicates        15   40.0%   60.0%    0.0%      0.0%       0
late_data         15   40.0%   60.0%   60.0%     60.0%       0
null_spike        15   33.3%   60.0%   60.0%     60.0%       0
schema_drift      18    0.0%  100.0%   16.7%     16.7%       0
dev               50   28.0%   70.0%   32.0%     32.0%       0
holdout           13   23.1%   76.9%   38.5%     38.5%       0

resolved 21/63 (95% CI 23 to 46%), fixes passing the hidden check 0/42

=== multi-1 ===
                   n    root   cause  action  resolved  tokens
overall           63   95.2%   96.8%  100.0%    100.0%   19985
duplicates        15  100.0%  100.0%  100.0%    100.0%   24870
late_data         15   93.3%   93.3%  100.0%    100.0%   24815
null_spike        15   93.3%  100.0%  100.0%    100.0%   18438
schema_drift      18   94.4%   94.4%  100.0%    100.0%   13178
dev           